In [1]:
import pandas as pd

# 1. Load movies.csv as movies dataframe and ratings.csv as ratings dataframe. Use info function to view the schema, null counts, and memory usage of each data frames

In [2]:
movies = pd.read_csv("https://raw.githubusercontent.com/abulbasar/data/master/movielens/movies.csv")
ratings = pd.read_csv("https://raw.githubusercontent.com/abulbasar/data/master/movielens/ratings.csv")

In [3]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


In [5]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 9125 entries, 0 to 9124
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9125 non-null   int64
 1   title    9125 non-null   str  
 2   genres   9125 non-null   str  
dtypes: int64(1), str(2)
memory usage: 582.8 KB


In [6]:
ratings.info()

<class 'pandas.DataFrame'>
RangeIndex: 100004 entries, 0 to 100003
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100004 non-null  int64  
 1   movieId    100004 non-null  int64  
 2   rating     100004 non-null  float64
 3   timestamp  100004 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


# 2. How many records are there in movies and ratings dataframe [movies: 9125, ratings: 100004


In [8]:
movies.shape

(9125, 3)

In [9]:
ratings.shape

(100004, 4)

# 3. Find the movies, for which title contains Godfather? [Hint: use movies.title.str.contains() to find substring in title]


In [13]:
movies[movies.title.str.lower().str.contains("godfather")]

,movieId,title,genres
695,858,"Godfather, The (1972)",Crime|Drama
977,1221,"Godfather: Part II, The (1974)",Crime|Drama
1585,2023,"Godfather: Part III, The (1990)",Crime|Drama|Mystery|Thriller
5470,8607,Tokyo Godfathers (2003),Adventure|Animation|Drama


# 4. How many ratings the movie "Godfather, The (1972)" has received?[Answer: 200]


In [17]:
movie_id = int(movies[movies.title == 'Godfather, The (1972)']['movieId'].values[0])
movie_id

858

In [19]:
ratings[ratings.movieId == movie_id].shape

(200, 4)

# 5. Using ratings dataframe, calculate average rating and number of ratings for each movie. What is the average rating for "Godfather, The (1972)" [Answer: 4.487]


In [22]:
avg_rating = ratings.groupby("movieId").rating.agg(["count", "mean"])
avg_rating

,count,mean
movieId,,
1,247,3.872470
2,107,3.401869
3,59,3.161017
4,13,2.384615
5,56,3.267857
...,...,...
161944,1,5.000000
162376,1,4.500000
162542,1,5.000000


In [24]:
avg_rating.loc[movie_id]

count    200.0000
mean       4.4875
Name: 858, dtype: float64

In [26]:
avg_rating.reset_index().query(f"movieId == {movie_id}")

,movieId,count,mean
695,858,200,4.4875


In [28]:
avg_rating[avg_rating.index == movie_id]

,count,mean
movieId,,
858,200,4.4875


# 6. Select all movie Ids that have received more than 100 ratings. How many movies are there having more than 100 ratings? [Answer: 149]

In [29]:
avg_rating = ratings.groupby("movieId").rating.agg(["count", "mean"]).query("count>100")
avg_rating

,count,mean
movieId,,
1,247,3.872470
2,107,3.401869
6,104,3.884615
10,122,3.450820
25,101,3.742574
...,...,...
7438,103,3.665049
8961,126,3.861111
33794,105,3.857143


In [30]:
avg_rating.shape

(149, 2)

# 7. From the output of step 5 and 6, take top 10 records with highest avg rating. [Hint: use sort and iloc] 

In [33]:
avg_rating = ratings.groupby("movieId").rating.agg(["count", "mean"]).query("count>100")
top10 = avg_rating.sort_values("mean", ascending=False).iloc[:10]
top10

,count,mean
movieId,,
858,200,4.487500
318,311,4.487138
1221,135,4.385185
50,201,4.370647
527,244,4.303279
1193,144,4.256944
608,224,4.256696
296,324,4.256173
2858,220,4.236364


# 8. Merge the output of 7 with movies dataframe to show title. [Hint: call pd.merge function]


In [34]:
avg_rating = ratings.groupby("movieId").rating.agg(["count", "mean"]).query("count>100")
top10 = avg_rating.sort_values("mean", ascending=False).iloc[:10]
pd.merge(top10, movies, on="movieId")

,movieId,count,mean,title,genres
0,858,200,4.487500,"Godfather, The (1972)",Crime|Drama
1,318,311,4.487138,"Shawshank Redemption, The (1994)",Crime|Drama
2,1221,135,4.385185,"Godfather: Part II, The (1974)",Crime|Drama
3,50,201,4.370647,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
4,527,244,4.303279,Schindler's List (1993),Drama|War
5,1193,144,4.256944,One Flew Over the Cuckoo's Nest (1975),Drama
6,608,224,4.256696,Fargo (1996),Comedy|Crime|Drama|Thriller
7,296,324,4.256173,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
8,2858,220,4.236364,American Beauty (1999),Drama|Romance
9,58559,121,4.235537,"Dark Knight, The (2008)",Action|Crime|Drama|IMAX


# 9. For each genre find top 3 movies. Genres column contain multiple values separated by pipe ('|'). Use explode (df.explode(...) function to generate one for each genre repeating the other values in the same row.

In [42]:
a = movies.copy()
a["genres"] = movies["genres"].str.split("|")
a = a.explode("genres")
a

,movieId,title,genres
0,1,Toy Story (1995),Adventure
0,1,Toy Story (1995),Animation
0,1,Toy Story (1995),Children
0,1,Toy Story (1995),Comedy
0,1,Toy Story (1995),Fantasy
...,...,...,...
9121,163056,Shin Godzilla (2016),Fantasy
9121,163056,Shin Godzilla (2016),Sci-Fi
9122,163949,The Beatles: Eight Days a Week - The Touring Y...,Documentary
9123,164977,The Gay Desperado (1936),Comedy


In [40]:
avg_rating = ratings.groupby("movieId").rating.agg(["count", "mean"]).query("count>100")
avg_rating

,count,mean
movieId,,
1,247,3.872470
2,107,3.401869
6,104,3.884615
10,122,3.450820
25,101,3.742574
...,...,...
7438,103,3.665049
8961,126,3.861111
33794,105,3.857143


In [44]:
b = avg_rating.merge(a, left_index=True, right_on="movieId")
b

,count,mean,movieId,title,genres
0,247,3.87247,1,Toy Story (1995),Adventure
0,247,3.87247,1,Toy Story (1995),Animation
0,247,3.87247,1,Toy Story (1995),Children
0,247,3.87247,1,Toy Story (1995),Comedy
0,247,3.87247,1,Toy Story (1995),Fantasy
...,...,...,...,...,...
7575,111,4.04955,79132,Inception (2010),Drama
7575,111,4.04955,79132,Inception (2010),Mystery
7575,111,4.04955,79132,Inception (2010),Sci-Fi
7575,111,4.04955,79132,Inception (2010),Thriller


In [51]:
b["rank"] = b.groupby("genres")["mean"].rank(ascending = False)
b = b[b['rank'] <= 3]
b.sort_values(["genres", "mean"], ascending=[True, False])

,count,mean,movieId,title,genres,rank
6916,121,4.235537,58559,"Dark Knight, The (2008)",Action,1.0
953,234,4.232906,1196,Star Wars: Episode V - The Empire Strikes Back...,Action,2.0
232,291,4.221649,260,Star Wars: Episode IV - A New Hope (1977),Action,3.0
953,234,4.232906,1196,Star Wars: Episode V - The Empire Strikes Back...,Adventure,1.0
912,145,4.224138,1136,Monty Python and the Holy Grail (1975),Adventure,2.0
232,291,4.221649,260,Star Wars: Episode IV - A New Hope (1977),Adventure,3.0
3805,130,3.884615,4886,"Monsters, Inc. (2001)",Animation,1.0
0,247,3.872470,1,Toy Story (1995),Animation,2.0
5626,126,3.861111,8961,"Incredibles, The (2004)",Animation,3.0
740,117,3.957265,919,"Wizard of Oz, The (1939)",Children,1.0
